# EVE 310 - Lab 09: Batch processing — many buildings

**Module 3 | 10/22/2026**

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ThyanRevolter/eve310-fall-2026/blob/main/labs/lab09-batch-processing/notebooks/lab09-multiple-files.ipynb)

## Learning objectives

By the end of this lab you will be able to:

1. List CSV files in a folder
2. Reuse the single-file analysis inside a for loop

## Before you start

1. Click **Copy to Drive** at the top of this window and work in the copy that opens. Colab throws away anything you did not copy when the runtime ends.
2. Run the setup cell below before anything else. It creates `DATA_DIR` and `FIGURES_DIR` and downloads this lab's data files.
3. Work down the notebook in order. Cells marked **Your turn** are the ones you complete.
4. When you are done: **Runtime > Restart session and run all**, then **File > Download > Download .ipynb**, and upload that file to Gradescope.


The analysis in the single-file notebook is unchanged. The new idea is to discover every `water_*.csv` in `DATA_DIR` and repeat the plot for each building.


## 0. Setup


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# EVE 310 setup - run this cell first, every time you open this notebook.
# It downloads this lab's data files into a "data" folder in your Colab session.
import pathlib
import urllib.request

LAB = "lab09-batch-processing"
DATA_FILES = [f"water_{code}.csv" for code in "AHG AND ARC ART BAT BEN BHD BLD BMA BME BUR CAL CBA CDL CLA CMA CPE CRD CRH DCP DFA EAS ECj EPS".split()]

DATA_DIR = pathlib.Path("data")
FIGURES_DIR = pathlib.Path("figures")
DATA_DIR.mkdir(exist_ok=True)
FIGURES_DIR.mkdir(exist_ok=True)

BASE_URL = f"https://raw.githubusercontent.com/ThyanRevolter/eve310-fall-2026/main/labs/{LAB}/data"
for name in DATA_FILES:
    if not (DATA_DIR / name).exists():
        urllib.request.urlretrieve(f"{BASE_URL}/{name}", DATA_DIR / name)

print(f"Ready. {len(DATA_FILES)} data file(s) in {DATA_DIR.resolve()}")

## 1. List CSV files


In [ ]:
csv_files = sorted(p for p in DATA_DIR.iterdir() if p.suffix.lower() == '.csv')
csv_files


## 2. Worked example — loop


In [ ]:
months = ['January', 'February', 'March', 'April', 'May', 'June',
          'July', 'August', 'September', 'October', 'November', 'December']
col = 'Water ( Gallons )'

for file_path in csv_files:
    water_df = pd.read_csv(file_path)
    building = file_path.stem.replace('water_', '')
    water_df['DateTime'] = pd.to_datetime(water_df['DateTime'])
    water_df['Month'] = water_df['DateTime'].dt.month
    w_std = np.std(water_df[col], ddof=1)
    w_mean = np.mean(water_df[col])
    upper, lower = w_mean + 3 * w_std, w_mean - 3 * w_std
    water_df = water_df.loc[(water_df[col] < upper) & (water_df[col] > lower)]
    water_df.boxplot(column=col, by='Month', grid=False, rot=45)
    plt.xticks(range(1, 13), months)
    plt.ylabel('Consumption (gallons)')
    plt.xlabel('Month')
    plt.title(f'{building} water consumption by month')
    plt.suptitle('')
    plt.savefig(FIGURES_DIR / f'{building}_water_boxplot.png', bbox_inches='tight', dpi=200)
    plt.close()
    print('saved', building)


## 3. Wrap-up

Check `FIGURES_DIR` for one box plot per building.
